### Evaluation

In [25]:
import os
import json
import subprocess

# Finished scripts
files = {
    d.name.replace("__", "/"): [
        list(json.load(open(f.path))["results"].keys())[0]
        for f in os.scandir(d)
        if f.is_file()
        if list(json.load(open(f.path))["results"].values())[0] != {}
    ]
    for d in os.scandir("logs")
}


def get_safe_filename(model: str, task: str) -> str:
    """Generate a safe filename for the script."""
    # Extract the model name after the slash and make it filesystem-safe
    model_name = model.split("/")[-1]
    # Replace any problematic characters
    model_name = model_name.replace("/", "_").replace(" ", "_")
    return f"eval_{model_name}_{task}.sh"


finished_scripts = [get_safe_filename(m, t) for m in files for t in files[m]]
print("Finished scripts:", len(finished_scripts))

# All scripts
all_scripts = [f.name for f in os.scandir("mv_exp/eval_scripts") if f.is_file()]
print("All scripts:", len(all_scripts))

# Running scripts
running_scripts = subprocess.check_output(
    ["squeue", "-u", "elganzory1", "-h", "-o", "%j"], text=True
)
running_scripts = [s.strip() + ".sh" for s in running_scripts.split("\n") if s.strip()]
print("Running scripts:", len(running_scripts))

# Remaining scripts
remaining_scripts = [
    s for s in all_scripts if s not in finished_scripts and s not in running_scripts
]
print("Remaining scripts:", len(remaining_scripts))

print()

string ="SCRIPTS=(\n"
for s in remaining_scripts:
    string += f'\t"{s}"\n'
string += ")"
print(string)

Finished scripts: 255
All scripts: 60
Running scripts: 0
Remaining scripts: 1

SCRIPTS=(
	"eval_1.7b-Comma0.1-300BT-WithChatTemplate_LiveCodeBench.sh"
)


### Chat Template

In [3]:
from transformers import AutoTokenizer

In [16]:
# model_id = "ali-elganzory/1.7b-Comma0.1-300BT-WithChatTemplate"
model_id = "ali-elganzory/ablation-model-fineweb-edu-WithChatTemplate"
model_id_sft = "ali-elganzory/" + model_id.split("/")[1].replace("-WithChatTemplate", "") + "-SFT-Tulu3-decontaminated"
t1 = AutoTokenizer.from_pretrained(model_id)
t2 = AutoTokenizer.from_pretrained(model_id_sft)

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [17]:
t1.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
    tokenize=False,
)

ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

In [18]:
t1.chat_template = t2.chat_template
t1.push_to_hub(model_id)

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ali-elganzory/ablation-model-fineweb-edu-WithChatTemplate/commit/b8b24e8461d728747e3d5f8db44e2f2b0ba3bd64', commit_message='Upload tokenizer', commit_description='', oid='b8b24e8461d728747e3d5f8db44e2f2b0ba3bd64', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ali-elganzory/ablation-model-fineweb-edu-WithChatTemplate', endpoint='https://huggingface.co', repo_type='model', repo_id='ali-elganzory/ablation-model-fineweb-edu-WithChatTemplate'), pr_revision=None, pr_num=None)

### Report

In [26]:
import os
import json
import pandas as pd
from IPython.display import display

json_files = [
    f
    for d in os.scandir("logs")
    for f in os.scandir(d)
    if f.is_file()
    if list(json.load(open(f.path))["results"].values())[0] != {}
]

# Metadata fields to exclude (not actual scores)
METADATA_FIELDS = {
    "num_examples",
    "completion_rate",
    "examples",
    "num_total",
    "solved_avg",
    "run_stats",
    "accuracy_std_err",
    "num_repeat",
    "accuracy_easy_avg",
    "accuracy_easy_std_err",
    "accuracy_medium_avg",
    "accuracy_medium_std_err",
    "accuracy_hard_avg",
    "accuracy_hard_std_err",
    "raw_metrics",
    "num_solved",
}

rows = []
for json_file in json_files:
    with open(json_file.path, "r") as f:
        data = json.load(f)

    model = data.get("model_name", "unknown")
    results = data.get("results", {})

    for task, metrics in results.items():
        for metric, score in metrics.items():
            if metric not in METADATA_FIELDS:
                rows.append(
                    {"model": model, "task": task, "metric": metric, "score": score}
                )

df_evalchemy = pd.DataFrame(rows)
print("Excluded metadata fields:")
print(len(df_evalchemy))
display(df_evalchemy.head(1))
print()


# add stage column that contains "SFT", "DPO", or "Base"
def detect_stage(model_name):
    if "SFT" in model_name:
        return "SFT"
    elif "DPO" in model_name:
        return "DPO"
    elif "Base" in model_name:
        return "Base"
    elif any(s.lower() in model_name.lower() for s in ["Instruct", "Qwen/Qwen3-1.7B"]):
        return "Instruct"
    else:
        return "Base"


df_evalchemy["stage"] = df_evalchemy["model"].apply(detect_stage)
print("Added stage column:")
display(df_evalchemy.head(1))
print()


def detect_decontaminated(model_name):
    if "decontaminated" in model_name.lower():
        return True
    else:
        return False


df_evalchemy["decontaminated"] = df_evalchemy["model"].apply(detect_decontaminated)
print("Added decontaminated column:")
display(df_evalchemy.head(1))
print()

# clean model name by keeping the part after the last "/" and removing "-SFT-Tulu3", "-DPO-Tulu3", or "-WithChatTemplate"
df_evalchemy_cleaned = df_evalchemy.copy()
df_evalchemy_cleaned["model"] = df_evalchemy_cleaned["model"].str.split("/").str[-1]
df_evalchemy_cleaned["model"] = df_evalchemy_cleaned["model"].str.replace(
    r"((-Base)?((-SFT|-DPO)-Tulu3(-decontaminated)?)?|-WithChatTemplate|-Instruct)",
    "",
    regex=True,
)
print("Cleaned model name:")
display(df_evalchemy_cleaned.head(1))
print()

# average score across metrics of the same task and model
df_evalchemy_avg = df_evalchemy_cleaned
df_evalchemy_avg = df_evalchemy_cleaned.groupby(["model", "task", "stage", "decontaminated"])
df_evalchemy_avg = df_evalchemy_avg.agg({"score": "mean"}).reset_index()
print(len(df_evalchemy_avg))
display(df_evalchemy_avg.head(3))

# save to csv
df_evalchemy_avg.to_csv("scores.csv", index=False)


Excluded metadata fields:
315


,model,task,metric,score
0,ali-elganzory/Qwen3-1.7B-Base-DPO-Tulu3-decont...,MBPP,pass@1,0.56



Added stage column:


,model,task,metric,score,stage
0,ali-elganzory/Qwen3-1.7B-Base-DPO-Tulu3-decont...,MBPP,pass@1,0.56,DPO



Added decontaminated column:


,model,task,metric,score,stage,decontaminated
0,ali-elganzory/Qwen3-1.7B-Base-DPO-Tulu3-decont...,MBPP,pass@1,0.56,DPO,True



Cleaned model name:


,model,task,metric,score,stage,decontaminated
0,Qwen3-1.7B,MBPP,pass@1,0.56,DPO,True



244


,model,task,stage,decontaminated,score
0,1.7b-Comma0.1-300BT,AIME24,Base,False,0.0
1,1.7b-Comma0.1-300BT,AIME24,DPO,True,0.0
2,1.7b-Comma0.1-300BT,AIME24,SFT,True,0.0


### LaTeX Table

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

# Configuration
MODEL_MAPPING = {
    # "1.7b-MixtureVitae-300BT-v1": "MV",
    "1.7b-MixtureVitae-300BT-v1-decontaminated": "MV",
    # "1.7b-MixtureVitae-300BT-v1-16k": "MV-16k",
    "1.7b-MixtureVitae-300BT-v1-decontaminated-16k": "MV-16k",
    "open-sci-ref-v0.01-1.7b-nemotron-hq-300B-16384": "OpenSci-NM",
    "SmolLM2-1.7B": "SmolLM2",
    "Qwen2.5-1.5B": "Qwen2.5",
    "Qwen3-1.7B": "Qwen3",
    "1.7b-Comma0.1-300BT": "Comma0.1",
    "ablation-model-fineweb-edu": "FineWeb-Edu",
}

# Models to display per stage (allows different models for different stages)
MODELS_PER_STAGE = {
    "Base": [
        "MV",
        # "MV-16k",
        # "OpenSci-NM",
        # "SmolLM2",
        # "Qwen2.5",
        # "Qwen3",
        "Comma0.1",
        "FineWeb-Edu",
    ],
    "SFT": [
        "MV",
        # "MV-16k",
        # "OpenSci-NM",
        # "SmolLM2",
        # "Qwen2.5",
        # "Qwen3",
        "Comma0.1",
        "FineWeb-Edu",
    ],
    "DPO": [
        "MV",
        # "MV-16k",
        # "OpenSci-NM",
        "Comma0.1",
        "FineWeb-Edu",
    ],
    # "Instruct": [
    #     "SmolLM2",
    #     "Qwen2.5",
    #     "Qwen3",
    # ],
}

STAGE_ORDER = [
    "Base",
    "SFT",
    "DPO",
    # "Instruct",
]

# Benchmark categories and order
BENCHMARK_CATEGORIES = {
    "Math": [
        "AIME24",
        "AIME25",
        "AMC23",
        "MATH500",
    ],
    "Code": [
        "HumanEval",
        "LiveCodeBench",
        "MBPP",
    ],
    "Sci": [
        "GPQADiamond",
        "JEEBench",
    ],
    "IF": [
        "IFEval",
    ],
}


def get_all_models() -> list:
    """Get all unique models across all stages, preserving order."""
    seen = set()
    all_models = []
    for stage in STAGE_ORDER:
        for model in MODELS_PER_STAGE.get(stage, []):
            if model not in seen:
                seen.add(model)
                all_models.append(model)
    return all_models


def load_and_filter_data(csv_path: str) -> pd.DataFrame:
    """Load CSV and filter to relevant models."""
    df = pd.read_csv(csv_path)

    # Map model names
    df["model_short"] = df["model"].map(MODEL_MAPPING)

    # Filter to only models in our mapping
    df = df[df["model_short"].notna()].copy()

    return df


def create_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Create pivot table with (benchmark) × (model, stage) structure."""
    pivot = df.pivot_table(
        index="task", columns=["model_short", "stage"], values="score", aggfunc="first"
    )

    # Reorder columns based on MODELS_PER_STAGE
    new_columns = []
    for stage in STAGE_ORDER:
        for model in MODELS_PER_STAGE.get(stage, []):
            if (model, stage) in pivot.columns:
                new_columns.append((model, stage))

    pivot = pivot.reindex(columns=new_columns)

    return pivot


def format_score(
    value: float, is_best_overall: bool = False, is_best_in_stage: bool = False, max_val: float = None
) -> str:
    """Format score as percentage with optional bold (overall best) and underline (stage best).
    If max_val is 0.0, do not bold or underline anything in this row or stage.
    """
    if pd.isna(value):
        return "--"

    # Convert to percentage and round to 1 decimal
    pct = value * 100

    # Format the number
    if pct == 0:
        formatted = "0.0"
    else:
        formatted = f"{pct:.1f}"

    # If max_val is provided and is 0.0, never bold or underline
    if max_val is not None and np.isclose(max_val, 0.0, rtol=1e-9, atol=1e-12):
        return formatted

    # Apply formatting: bold for overall best, underline for stage best
    if is_best_overall:
        formatted = f"\\textbf{{{formatted}}}"
    if is_best_in_stage:
        formatted = f"\\underline{{{formatted}}}"

    return formatted


def find_best_in_row(row: pd.Series) -> set:
    """Find indices of best (maximum) values in a row."""
    valid_values = row.dropna()
    if len(valid_values) == 0:
        return set()

    max_val = valid_values.max()
    # Handle floating point comparison
    best_indices = set()
    for idx, val in row.items():
        if not pd.isna(val) and np.isclose(val, max_val, rtol=1e-9):
            best_indices.add(idx)

    return best_indices


def find_best_per_stage(row: pd.Series) -> dict:
    """Find best (maximum) values within each stage."""
    best_per_stage = {}

    for stage in STAGE_ORDER:
        models = MODELS_PER_STAGE.get(stage, [])
        stage_values = {}

        for model in models:
            col = (model, stage)
            if col in row.index and not pd.isna(row[col]):
                stage_values[col] = row[col]

        if stage_values:
            max_val = max(stage_values.values())
            best_per_stage[stage] = {
                col
                for col, val in stage_values.items()
                if np.isclose(val, max_val, rtol=1e-9)
            }
        else:
            best_per_stage[stage] = set()

    return best_per_stage


def calculate_averages(pivot: pd.DataFrame) -> pd.Series:
    """Calculate column averages."""
    return pivot.mean(skipna=True)


def get_decontaminated_combinations(df: pd.DataFrame) -> set:
    """Extract (model_short, stage) combinations that are decontaminated."""
    decontaminated_rows = df[df["decontaminated"] == True]  # noqa: E712
    combinations = set(
        zip(decontaminated_rows["model_short"], decontaminated_rows["stage"])
    )
    return combinations


def get_model_header(model: str, stage: str, decontaminated_combinations: set) -> str:
    """Get model header. (No longer handles decontaminated annotation.)"""
    # Always return model without the decontamination marker, since all are decontaminated.
    return f"\\rotatebox{{45}}{{{model}}}"


def generate_latex_table(
    pivot: pd.DataFrame,
    decontaminated_combinations: set = None,
    caption: str = None,
    label: str = None,
) -> str:
    """Generate the full LaTeX table code."""
    if decontaminated_combinations is None:
        decontaminated_combinations = set()

    lines = []
    num_stages = len(STAGE_ORDER)

    # File header
    lines.append("""
\\documentclass{article}
\\usepackage[margin=0.5in]{geometry}
\\usepackage{booktabs}
\\usepackage{graphicx}
\\usepackage{multirow}
\\usepackage{makecell}

\\begin{document}
""")

    # Table header
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")

    if caption:
        lines.append(f"\\caption{{{caption}}}")
    else:
        lines.append(
            r"\caption{Evaluation results of models post-trained with SFT and DPO using Tulu3's recipe.}"
        )

    if label:
        lines.append(f"\\label{{{label}}}")
    else:
        lines.append(r"\label{tab:evalchemy}")

    # Start resizebox to fit page width
    lines.append(r"\resizebox{\textwidth}{!}{%")

    # Column specification: cl | ccc... | ccc... (variable per stage)
    col_groups = []
    for stage in STAGE_ORDER:
        num_models = len(MODELS_PER_STAGE.get(stage, []))
        col_groups.append("c" * num_models)
    col_spec = " | ".join(col_groups)
    lines.append(f"\\begin{{tabular}}{{cl | {col_spec}}}")
    lines.append(r"\toprule")

    # Header row 1: Stage names with multicolumn
    stage_headers = []
    for i, stage in enumerate(STAGE_ORDER):
        num_models = len(MODELS_PER_STAGE.get(stage, []))
        separator = "|" if i < num_stages - 1 else ""
        stage_headers.append(
            f"\\multicolumn{{{num_models}}}{{c{separator}}}{{\\textbf{{{stage}}}}}"
        )
    lines.append(f" & & {' & '.join(stage_headers)} \\\\")

    # cmidrule for each stage group
    cmidrules = []
    start_col = 3  # First data column (after row label and benchmark name)
    for stage in STAGE_ORDER:
        num_models = len(MODELS_PER_STAGE.get(stage, []))
        end_col = start_col + num_models - 1
        cmidrules.append(f"\\cmidrule(lr){{{start_col}-{end_col}}}")
        start_col = end_col + 1
    lines.append(" ".join(cmidrules))

    # Header row 2: Model names (no footnotes for decontaminated)
    model_headers = []
    for stage in STAGE_ORDER:
        for model in MODELS_PER_STAGE.get(stage, []):
            model_headers.append(
                get_model_header(model, stage, decontaminated_combinations)
            )

    lines.append(f" & \\textbf{{Benchmark}} & {' & '.join(model_headers)} \\\\")
    lines.append(r"\midrule")

    # Calculate averages and find best (overall and per-stage)
    averages = calculate_averages(pivot)
    avg_best_overall = find_best_in_row(averages)
    # Find max average for masking bold/underline if all are zero
    avg_max_val = averages.max() if len(averages) > 0 else None
    avg_best_per_stage = {}
    avg_max_per_stage = {}
    for stage in STAGE_ORDER:
        models = MODELS_PER_STAGE.get(stage, [])
        stage_values = []
        for model in models:
            col = (model, stage)
            if col in averages.index:
                stage_values.append(averages[col])
        if stage_values:
            stage_max = max(stage_values)
            avg_max_per_stage[stage] = stage_max
        else:
            avg_max_per_stage[stage] = None
    avg_best_per_stage = find_best_per_stage(averages)

    # Average row
    avg_values = []
    for stage in STAGE_ORDER:
        for model in MODELS_PER_STAGE.get(stage, []):
            col = (model, stage)
            if col in averages.index:
                is_best_overall = col in avg_best_overall
                is_best_in_stage = col in avg_best_per_stage.get(stage, set())
                # If the overall max_val is 0.0, don't bold/underline
                val_max_for_this = avg_max_per_stage.get(stage, None)
                val_max_overall = avg_max_val if avg_max_val is not None else 0.0
                max_is_zero = (
                    (val_max_for_this is not None and np.isclose(val_max_for_this, 0.0, rtol=1e-9, atol=1e-12))
                    or (val_max_overall is not None and np.isclose(val_max_overall, 0.0, rtol=1e-9, atol=1e-12))
                )
                # The format_score checks for max_val==0.0 so pass it
                max_val_to_check = val_max_for_this if val_max_for_this is not None else val_max_overall
                avg_values.append(
                    format_score(averages[col], is_best_overall, is_best_in_stage, max_val_to_check)
                )
            else:
                avg_values.append("--")

    lines.append(f" & \\textbf{{Average}} & {' & '.join(avg_values)} \\\\")
    lines.append(r"\midrule")

    # Data rows by category
    for cat_idx, (category, benchmarks) in enumerate(BENCHMARK_CATEGORIES.items()):
        # Filter benchmarks that exist in the data
        existing_benchmarks = [b for b in benchmarks if b in pivot.index]

        if not existing_benchmarks:
            continue

        num_rows = len(existing_benchmarks)

        for i, benchmark in enumerate(existing_benchmarks):
            row = pivot.loc[benchmark]
            best_overall = find_best_in_row(row)
            best_per_stage = find_best_per_stage(row)
            # Compute row max for this benchmark
            row_max_val = row.max() if (~row.isna()).any() else None
            # Max values for each stage in this row
            row_max_per_stage = {}
            for stage in STAGE_ORDER:
                models = MODELS_PER_STAGE.get(stage, [])
                stage_vals = []
                for model in models:
                    col = (model, stage)
                    if col in row.index and not pd.isna(row[col]):
                        stage_vals.append(row[col])
                if stage_vals:
                    stage_max = max(stage_vals)
                    row_max_per_stage[stage] = stage_max
                else:
                    row_max_per_stage[stage] = None

            # Format values
            values = []
            for stage in STAGE_ORDER:
                for model in MODELS_PER_STAGE.get(stage, []):
                    col = (model, stage)
                    if col in row.index:
                        is_best_overall = col in best_overall
                        is_best_in_stage = col in best_per_stage.get(stage, set())
                        val_max_for_this = row_max_per_stage.get(stage, None)
                        val_max_overall = row_max_val if row_max_val is not None else 0.0
                        # Don't bold/underline if max is 0.0 at row or stage level
                        max_is_zero = (
                            (val_max_for_this is not None and np.isclose(val_max_for_this, 0.0, rtol=1e-9, atol=1e-12))
                            or (val_max_overall is not None and np.isclose(val_max_overall, 0.0, rtol=1e-9, atol=1e-12))
                        )
                        # The format_score checks for max_val==0.0 so pass it
                        max_val_to_check = val_max_for_this if val_max_for_this is not None else val_max_overall
                        values.append(
                            format_score(row[col], is_best_overall, is_best_in_stage, max_val_to_check)
                        )
                    else:
                        values.append("--")

            # Build row string
            if i == 0:
                # First row of category - add rotated category label
                row_str = f"\\multirow{{{num_rows}}}{{*}}{{\\rotatebox{{90}}{{\\textit{{{category}}}}}}}"
            else:
                row_str = ""

            row_str += f" & {benchmark} & {' & '.join(values)} \\\\"
            lines.append(row_str)

        # Add midrule after each category except the last
        if cat_idx < len(BENCHMARK_CATEGORIES) - 1:
            lines.append(r"\midrule")

    # Table footer
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}")  # End resizebox

    # Two-column footer: model mapping on left, formatting notes on right (no decontaminated note)
    lines.append(r"\vspace{0.5em}")
    lines.append(r"\noindent")
    lines.append(r"\begin{minipage}[t]{0.6\textwidth}")
    lines.append(r"\raggedright")

    # Add model name mapping legend (only for models with different short names)
    all_used_models = get_all_models()
    mapping_entries = [
        (short, full)
        for full, short in MODEL_MAPPING.items()
        if full != short and short in all_used_models
    ]
    if mapping_entries:
        lines.append(r"\small")
        lines.append(r"\begin{tabular}{r@{: }l}")
        for short, full in mapping_entries:
            lines.append(f"\\textbf{{{short}}} & {full} \\\\")
        lines.append(r"\end{tabular}")

    # (No decontaminated footnote of any kind)

    lines.append(r"\end{minipage}%")
    lines.append(r"\hfill")
    lines.append(r"\begin{minipage}[t]{0.35\textwidth}")
    lines.append(r"\raggedleft")
    lines.append(r"\footnotesize{")

    # Note about formatting (each on separate line)
    lines.append(r"\underline{Underlined}: best within stage.\\")
    lines.append(r"\textbf{Bold}: best overall.")

    lines.append(r"}")
    lines.append(r"\end{minipage}")

    lines.append(r"\end{table}")

    # File footer
    lines.append("""
\\end{document}
""")

    return "\n".join(lines)


csv_path = "scores.csv"
output_path = "scores.tex"

# Load data
print(f"Loading data from {csv_path}")
df = load_and_filter_data(csv_path)

print(f"Found {len(df)} rows for models: {df['model_short'].unique().tolist()}")

# Create pivot table
pivot = create_pivot_table(df)

print(f"Pivot table shape: {pivot.shape}")
print(f"Benchmarks: {pivot.index.tolist()}")

# Get decontaminated combinations
decontaminated = get_decontaminated_combinations(df)
if decontaminated:
    print(f"Decontaminated combinations: {decontaminated}")

# Print models per stage
print("\nModels per stage:")
for stage in STAGE_ORDER:
    print(f"  {stage}: {MODELS_PER_STAGE.get(stage, [])}")

# Generate LaTeX
latex_code = generate_latex_table(pivot, decontaminated_combinations=decontaminated)

# Save to file
with open(output_path, "w") as f:
    f.write(latex_code)

print(f"\nLaTeX table saved to {output_path}")
print("\n" + "=" * 60)
print("Generated LaTeX code:")
print("=" * 60)
print(latex_code)

Loading data from scores.csv
Found 133 rows for models: ['Comma0.1', 'MV', 'MV-16k', 'Qwen2.5', 'Qwen3', 'SmolLM2', 'FineWeb-Edu']
Pivot table shape: (10, 9)
Benchmarks: ['AIME24', 'AIME25', 'AMC23', 'GPQADiamond', 'HumanEval', 'IFEval', 'JEEBench', 'LiveCodeBench', 'MATH500', 'MBPP']
Decontaminated combinations: {('Comma0.1', 'SFT'), ('FineWeb-Edu', 'DPO'), ('MV-16k', 'Base'), ('MV', 'SFT'), ('Qwen3', 'DPO'), ('Qwen2.5', 'DPO'), ('FineWeb-Edu', 'SFT'), ('MV-16k', 'DPO'), ('Comma0.1', 'DPO'), ('MV', 'Base'), ('MV', 'DPO'), ('SmolLM2', 'DPO'), ('MV-16k', 'SFT')}

Models per stage:
  Base: ['MV', 'Comma0.1', 'FineWeb-Edu']
  SFT: ['MV', 'Comma0.1', 'FineWeb-Edu']
  DPO: ['MV', 'Comma0.1', 'FineWeb-Edu']

LaTeX table saved to scores.tex

Generated LaTeX code:

\documentclass{article}
\usepackage[margin=0.5in]{geometry}
\usepackage{booktabs}
\usepackage{graphicx}
\usepackage{multirow}
\usepackage{makecell}

\begin{document}

\begin{table}[htbp]
\centering
\caption{Evaluation results of mod